# Recommendation of Anime

*Dataset cleaning and preparation*

In [25]:
import pandas as pd

animes = pd.read_csv("/Users/mac/Desktop/personalisation/personalisation-23-24-main/mini/data/animes.csv")

animes = animes.drop(columns=[animes.columns[0]])

new_title_column = animes.columns[0]

animes.columns = ['title'] + list(animes.columns[1:])

animes = animes.drop_duplicates(subset=new_title_column, keep='first')

animes = animes.dropna()

animes


,title,synopsis,genre,aired,episodes,members,popularity,ranked,score,img_url,link
0,Haikyuu!! Second Season,Following their participation at the Inter-Hig...,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun...","Oct 4, 2015 to Mar 27, 2016",25.0,489888,141,25.0,8.82,https://cdn.myanimelist.net/images/anime/9/766...,https://myanimelist.net/anime/28891/Haikyuu_Se...
1,Shigatsu wa Kimi no Uso,Music accompanies the path of the human metron...,"['Drama', 'Music', 'Romance', 'School', 'Shoun...","Oct 10, 2014 to Mar 20, 2015",22.0,995473,28,24.0,8.83,https://cdn.myanimelist.net/images/anime/3/671...,https://myanimelist.net/anime/23273/Shigatsu_w...
2,Made in Abyss,The Abyss—a gaping chasm stretching down into ...,"['Sci-Fi', 'Adventure', 'Mystery', 'Drama', 'F...","Jul 7, 2017 to Sep 29, 2017",13.0,581663,98,23.0,8.83,https://cdn.myanimelist.net/images/anime/6/867...,https://myanimelist.net/anime/34599/Made_in_Abyss
3,Fullmetal Alchemist: Brotherhood,"""In order for something to be obtained, someth...","['Action', 'Military', 'Adventure', 'Comedy', ...","Apr 5, 2009 to Jul 4, 2010",64.0,1615084,4,1.0,9.23,https://cdn.myanimelist.net/images/anime/1223/...,https://myanimelist.net/anime/5114/Fullmetal_A...
4,Kizumonogatari III: Reiketsu-hen,After helping revive the legendary vampire Kis...,"['Action', 'Mystery', 'Supernatural', 'Vampire']","Jan 6, 2017",1.0,214621,502,22.0,8.83,https://cdn.myanimelist.net/images/anime/3/815...,https://myanimelist.net/anime/31758/Kizumonoga...
...,...,...,...,...,...,...,...,...,...,...,...
19002,Naruto x UT,All-new animation offered throughout UNIQLO cl...,"['Action', 'Comedy', 'Super Power', 'Martial A...","Jan 1, 2011",1.0,34155,2382,1728.0,7.50,https://cdn.myanimelist.net/images/anime/3/304...,https://myanimelist.net/anime/10075/Naruto_x_UT
19003,Miira no Kaikata,High school student Sora Kashiwagi is accustom...,"['Slice of Life', 'Comedy', 'Supernatural']","Jan 12, 2018 to Mar 30, 2018",12.0,61459,1648,1727.0,7.50,https://cdn.myanimelist.net/images/anime/1486/...,https://myanimelist.net/anime/35828/Miira_no_K...
19004,Shinryaku!? Ika Musume,"After regaining her squid-like abilities, Ika ...","['Slice of Life', 'Comedy', 'Shounen']","Sep 27, 2011 to Dec 25, 2011",12.0,67422,1547,1548.0,7.56,https://cdn.myanimelist.net/images/anime/6/301...,https://myanimelist.net/anime/10378/Shinryaku_...
19005,Kingsglaive: Final Fantasy XV,"For years, the Niflheim Empire and the kingdom...",['Action'],"Jul 9, 2016",1.0,41077,2154,1544.0,7.56,https://cdn.myanimelist.net/images/anime/12/79...,https://myanimelist.net/anime/33082/Kingsglaiv...


In [26]:
genre_column = 'genre'  

def get_unique_genres():
    if genre_column in animes.columns:

        animes[genre_column] = animes[genre_column].str.replace('[\[\]]', '', regex=True)
        
        all_genres = animes[genre_column].str.split(',').explode().str.strip()
        
        unique_genres = all_genres.drop_duplicates().dropna().values
        
        return unique_genres
    
    else:
        
        return f"The genre column '{genre_column}' does not exist in the dataset."

unique_genres = get_unique_genres()

print("all kinds of genres:")
print(unique_genres)



all kinds of genres:
["'Comedy'" "'Sports'" "'Drama'" "'School'" "'Shounen'" "'Music'"
 "'Romance'" "'Sci-Fi'" "'Adventure'" "'Mystery'" "'Fantasy'" "'Action'"
 "'Military'" "'Magic'" "'Supernatural'" "'Vampire'" "'Slice of Life'"
 "'Demons'" "'Historical'" "'Super Power'" "'Mecha'" "'Parody'"
 "'Samurai'" "'Seinen'" "'Police'" "'Psychological'" "'Josei'" "'Space'"
 "'Kids'" "'Shoujo Ai'" "'Ecchi'" "'Shoujo'" "'Horror'" "'Shounen Ai'"
 "'Cars'" "'Martial Arts'" "'Game'" "'Thriller'" "'Dementia'" "'Harem'" '']


In [27]:
title_column = 'title'

def get_genre_by_title(title):
    if title_column in animes.columns and genre_column in animes.columns:

        matching_row = animes[animes[title_column] == title]
        
        if not matching_row.empty:

            return matching_row[genre_column].values[0]
        else:

            return f"No anime found with title '{title}'"
        
    else:
        
        return f"Either the title column '{title_column}' or the genre column '{genre_column}' does not exist in the dataset."


title = "InuYasha Movie 2: Kagami no Naka no Mugenjo"  
genre = get_genre_by_title(title)

print("genre of anime " + title + ":")
print(genre)

genre of anime InuYasha Movie 2: Kagami no Naka no Mugenjo:
'Action', 'Adventure', 'Comedy', 'Historical', 'Demons', 'Supernatural', 'Drama', 'Magic', 'Romance', 'Fantasy', 'Shounen'


*Recommendation method 1*

Based on numerical value

In [28]:
features = ["title", "episodes", "popularity", "ranked", "score"]
subset_features = animes[features].dropna()
subset_features.describe()

,episodes,popularity,ranked,score
count,13675.000000,13675.000000,13675.000000,13675.000000
mean,12.523583,7999.478391,7108.839269,6.379982
std,49.247172,4844.068308,4167.194733,0.989704
min,1.000000,1.000000,1.000000,1.900000
25%,1.000000,3563.500000,3499.500000,5.710000
50%,1.000000,8091.000000,7070.000000,6.410000
75%,12.000000,12263.500000,10591.500000,7.110000
max,3057.000000,16320.000000,14675.000000,9.230000


In [29]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

numeric_features = subset_features.select_dtypes(include=['int64', 'float64'])

scaled_features = StandardScaler().fit_transform(numeric_features)

similarities = cosine_similarity(scaled_features)

similarities_df = pd.DataFrame(similarities, index=subset_features['title'], columns=subset_features['title'])

similarities_df


title,Haikyuu!! Second Season,Shigatsu wa Kimi no Uso,Made in Abyss,Fullmetal Alchemist: Brotherhood,Kizumonogatari III: Reiketsu-hen,Mob Psycho 100 II,Sen to Chihiro no Kamikakushi,Kimetsu no Yaiba,Owarimonogatari 2nd Season,Code Geass: Hangyaku no Lelouch R2,...,Yowamushi Pedal: Re:RIDE,Pokemon Movie 02: Maboroshi no Pokemon Lugia Bakutan,Mekakucity V's,Amagami SS,Net-juu no Susume Special,Naruto x UT,Miira no Kaikata,Shinryaku!? Ika Musume,Kingsglaive: Final Fantasy XV,Chuunibyou demo Koi ga Shitai!: Kirameki no... Slapstick Noel
title,,,,,,,,,,,,,,,,,,,,,
Haikyuu!! Second Season,1.000000,0.999823,0.997449,0.977793,0.989524,0.997335,0.989918,0.999816,0.993517,0.999822,...,0.948165,0.935483,0.912128,0.945225,0.948562,0.955904,0.963555,0.966824,0.958392,0.947633
Shigatsu wa Kimi no Uso,0.999823,1.000000,0.998573,0.973962,0.991841,0.998422,0.992298,0.999543,0.995128,0.999677,...,0.950894,0.939379,0.913345,0.945315,0.952171,0.959258,0.965259,0.968476,0.961671,0.951227
Made in Abyss,0.997449,0.998573,1.000000,0.961243,0.997176,0.999891,0.997470,0.996984,0.998572,0.997431,...,0.959772,0.945666,0.921385,0.939928,0.958836,0.966372,0.966441,0.969702,0.968556,0.957365
Fullmetal Alchemist: Brotherhood,0.977793,0.973962,0.961243,1.000000,0.940126,0.962107,0.940962,0.979735,0.952731,0.978514,...,0.876403,0.859983,0.853171,0.911787,0.874220,0.882465,0.909448,0.913429,0.886235,0.874760
Kizumonogatari III: Reiketsu-hen,0.989524,0.991841,0.997176,0.940126,1.000000,0.997308,0.999795,0.988971,0.999085,0.989792,...,0.968619,0.945637,0.932498,0.923751,0.960385,0.969585,0.960684,0.964206,0.971479,0.957811
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Naruto x UT,0.955904,0.959258,0.966372,0.882465,0.969585,0.963084,0.969131,0.950674,0.961795,0.951830,...,0.980656,0.990722,0.923054,0.962019,0.997770,1.000000,0.992546,0.993132,0.999961,0.995670
Miira no Kaikata,0.963555,0.965259,0.966441,0.909448,0.960684,0.962580,0.961477,0.958324,0.956069,0.959007,...,0.959692,0.991705,0.893813,0.987796,0.994564,0.992546,1.000000,0.999895,0.992936,0.994809
Shinryaku!? Ika Musume,0.966824,0.968476,0.969702,0.913429,0.964206,0.966040,0.964850,0.961808,0.959886,0.962462,...,0.962593,0.990585,0.898987,0.986388,0.994323,0.993132,0.999895,1.000000,0.993593,0.994320


In [30]:
top_30_df = similarities_df.iloc[:30, :30]

styled_df = top_30_df.style.background_gradient(cmap='Blues')

styled_df

title,Haikyuu!! Second Season,Shigatsu wa Kimi no Uso,Made in Abyss,Fullmetal Alchemist: Brotherhood,Kizumonogatari III: Reiketsu-hen,Mob Psycho 100 II,Sen to Chihiro no Kamikakushi,Kimetsu no Yaiba,Owarimonogatari 2nd Season,Code Geass: Hangyaku no Lelouch R2,Haikyuu!!: Karasuno Koukou vs. Shiratorizawa Gakuen Koukou,Gintama.,Gintama Movie 2: Kanketsu-hen - Yorozuya yo Eien Nare,Gintama,Clannad: After Story,Gintama': Enchousen,One Punch Man,Kaguya-hime no Monogatari,Koukaku Kidoutai 2.0,Nodame Cantabile: Finale - Mine to Kiyora no Saikai,Saraiya Goyou,Saint Seiya: Meiou Hades Meikai-hen,Noragami OVA,Lupin III: Part II,"Kobayashi-san Chi no Maid Dragon: Valentine, Soshite Onsen! - Amari Kitai Shinaide Kudasai",Kuroko no Basket 2nd Season NG-shuu,K-On!: Live House!,K-On!,InuYasha Movie 3: Tenka Hadou no Ken,Haikyuu!! Movie 2: Shousha to Haisha
title,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Haikyuu!! Second Season,1.000000,0.999823,0.997449,0.977793,0.989524,0.997335,0.989918,0.999816,0.993517,0.999822,0.995700,0.995630,0.988359,0.729683,0.999601,0.996067,0.996441,0.983724,0.978367,0.972434,0.982238,0.983331,0.967085,0.675403,0.969862,0.981036,0.969120,0.972800,0.970936,0.971278
Shigatsu wa Kimi no Uso,0.999823,1.000000,0.998573,0.973962,0.991841,0.998422,0.992298,0.999543,0.995128,0.999677,0.997077,0.996649,0.990535,0.717151,0.999549,0.997003,0.997747,0.986276,0.981049,0.975055,0.983509,0.984334,0.970099,0.661740,0.972843,0.982543,0.972139,0.974420,0.973840,0.973845
Made in Abyss,0.997449,0.998573,1.000000,0.961243,0.997176,0.999891,0.997470,0.996984,0.998572,0.997431,0.999507,0.998574,0.995917,0.679062,0.997661,0.998604,0.999409,0.992065,0.987111,0.981698,0.985070,0.986108,0.975464,0.621470,0.978940,0.985606,0.978163,0.975169,0.980074,0.980639
Fullmetal Alchemist: Brotherhood,0.977793,0.973962,0.961243,1.000000,0.940126,0.962107,0.940962,0.979735,0.952731,0.978514,0.957822,0.961174,0.941516,0.849949,0.977651,0.963020,0.956784,0.925395,0.916271,0.907929,0.933426,0.936112,0.901454,0.799418,0.903393,0.928613,0.902492,0.922881,0.904777,0.906773
Kizumonogatari III: Reiketsu-hen,0.989524,0.991841,0.997176,0.940126,1.000000,0.997308,0.999795,0.988971,0.999085,0.989792,0.998358,0.997261,0.999328,0.624604,0.990610,0.996805,0.996510,0.994875,0.990044,0.985985,0.981465,0.983700,0.975823,0.564757,0.981027,0.984763,0.979983,0.968837,0.982669,0.985401
Mob Psycho 100 II,0.997335,0.998422,0.999891,0.962107,0.997308,1.000000,0.997530,0.997133,0.999028,0.997563,0.999783,0.999192,0.996583,0.679233,0.997924,0.999250,0.998801,0.990858,0.985375,0.979983,0.982704,0.984296,0.972373,0.620715,0.976302,0.983684,0.975421,0.971791,0.977632,0.979056
Sen to Chihiro no Kamikakushi,0.989918,0.992298,0.997470,0.940962,0.999795,0.997530,1.000000,0.989395,0.998888,0.990270,0.998501,0.996915,0.998830,0.626196,0.991039,0.996589,0.996909,0.994242,0.989260,0.984243,0.980941,0.982203,0.976460,0.564592,0.980832,0.983405,0.979972,0.970124,0.982118,0.983338
Kimetsu no Yaiba,0.999816,0.999543,0.996984,0.979735,0.988971,0.997133,0.989395,1.000000,0.993521,0.999976,0.995631,0.995910,0.988423,0.732743,0.999889,0.996441,0.995333,0.981263,0.975152,0.968943,0.978575,0.980111,0.962534,0.676927,0.965625,0.977623,0.964802,0.968367,0.966848,0.967860
Owarimonogatari 2nd Season,0.993517,0.995128,0.998572,0.952731,0.999085,0.999028,0.998888,0.993521,1.000000,0.994099,0.999688,0.999477,0.999238,0.652794,0.994884,0.999293,0.996935,0.991443,0.985547,0.981080,0.979034,0.981998,0.969609,0.592740,0.975034,0.982001,0.973873,0.965341,0.976893,0.980595


In [31]:
animes.set_index('title', inplace=True)
animes.index.values

array(['Haikyuu!! Second Season', 'Shigatsu wa Kimi no Uso',
       'Made in Abyss', ..., 'Shinryaku!? Ika Musume',
       'Kingsglaive: Final Fantasy XV',
       'Chuunibyou demo Koi ga Shitai!: Kirameki no... Slapstick Noel'],
      dtype=object)

In [47]:
import numpy as np

anime = 'InuYasha Movie 2: Kagami no Naka no Mugenjo' 

sorted_indices = np.argsort(similarities[animes.index.get_loc(anime)])

top_n_similar_indices = sorted_indices[1:n+1]

top_n_similar = animes.index[top_n_similar_indices]

print(top_n_similar)

Index(['Lalala Lala-chan', 'Fuusen Inu Tinny 2nd Season',
       'Usagi no Mofy (TV 2016)', 'Mori no Senshi Bonolon',
       'Bemubemu Hunter Kotengu Tenmaru', 'Biriken Nandemo Shoukai',
       'Zhu Zhu Xia: Fan Wai - Pin Zhuang Tegong Dui', 'Great Hunt',
       'Kensaku to Enjin no ABC Days', 'Ganbare! Lulu Lolo'],
      dtype='object', name='title')


In [51]:
anime_title1 = 'Lalala Lala-chan'
row_data1 = animes.loc[anime_title1]

anime_title2 = 'Great Hunt'
row_data2 = animes.loc[anime_title2]

print(row_data1)
print(row_data2)


synopsis      Lala-chan is a slightly selfish squirrel girl....
genre                            'Adventure', 'Fantasy', 'Kids'
aired                              Mar 21, 2015 to Feb 26, 2016
episodes                                                   24.0
members                                                     120
popularity                                                14920
ranked                                                  12864.0
score                                                      4.93
img_url       https://cdn.myanimelist.net/images/anime/1785/...
link          https://myanimelist.net/anime/36549/Lalala_Lal...
Name: Lalala Lala-chan, dtype: object
synopsis      Great Hunt is a mysterious amalgamation of hea...
genre                                         'Comedy', 'Music'
aired                                      Apr 10, 2008 to 2008
episodes                                                   19.0
members                                                     156
po

*Recommendation method 2*

Based on type and content

In [30]:
animes = pd.read_csv("/Users/mac/Desktop/personalisation/personalisation-23-24-main/mini/data/animes.csv")

animes = animes.drop(columns=[animes.columns[0]])

new_title_column = animes.columns[0]
animes.columns = ['title'] + list(animes.columns[1:])

animes = animes.drop_duplicates(subset=new_title_column, keep='first')

animes = animes.dropna()

df_exploded = animes.assign(genres=animes['genre'].str.split(',')).explode('genre')
df_encoded = pd.get_dummies(df_exploded, columns=['genre'])
one_hot_animes = df_encoded.groupby('title').sum().reset_index()
one_hot_animes.index = one_hot_animes["title"].values

one_hot_animes

,title,synopsis,aired,episodes,members,popularity,ranked,score,img_url,link,...,"genre_['Supernatural', 'Music']","genre_['Supernatural', 'School']","genre_['Supernatural', 'Shounen']","genre_['Supernatural', 'Vampire']",genre_['Supernatural'],"genre_['Thriller', 'Mystery', 'Sci-Fi']","genre_['Thriller', 'Sci-Fi']",genre_['Thriller'],genre_['Vampire'],genre_[]
"""0""","""0""",This music video tells how a shy girl with a s...,"Oct 23, 2013",1.00,2594,7345,"11,123.00",4.77,https://cdn.myanimelist.net/images/anime/6/548...,https://myanimelist.net/anime/20707/0,...,0,0,0,0,0,0,0,0,0,0
"""Aesop"" no Ohanashi yori: Ushi to Kaeru, Yokubatta Inu","""Aesop"" no Ohanashi yori: Ushi to Kaeru, Yokub...",Based on Aesop's Fables.,"Mar 21, 1970",1.00,273,12413,"9,560.00",5.61,https://cdn.myanimelist.net/images/anime/3/651...,https://myanimelist.net/anime/25627/Aesop_no_O...,...,0,0,0,0,0,0,0,0,0,0
"""Bungaku Shoujo"" Kyou no Oyatsu: Hatsukoi","""Bungaku Shoujo"" Kyou no Oyatsu: Hatsukoi",Short episode bundled with the limited edition...,"Dec 26, 2009",1.00,16961,3466,"3,923.00",6.96,https://cdn.myanimelist.net/images/anime/2/799...,https://myanimelist.net/anime/7669/Bungaku_Sho...,...,0,0,0,0,0,0,0,0,0,0
"""Bungaku Shoujo"" Memoire","""Bungaku Shoujo"" Memoire",Episodes which depict the background stories o...,"Jun 25, 2010 to Dec 24, 2010",3.00,23504,2943,"2,087.00",7.40,https://cdn.myanimelist.net/images/anime/6/267...,https://myanimelist.net/anime/8481/Bungaku_Sho...,...,0,0,0,0,0,0,0,0,0,0
"""Bungaku Shoujo"" Movie","""Bungaku Shoujo"" Movie","The protagonist of the story, Konoha Inoue, is...","May 1, 2010",1.00,53766,1799,"1,778.00",7.48,https://cdn.myanimelist.net/images/anime/8/811...,https://myanimelist.net/anime/6408/Bungaku_Sho...,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
xxxHOLiC Rou,xxxHOLiC Rou,10 years after the events of xxxHOLiC Shunmuk...,"Apr 23, 2010 to Mar 9, 2011",2.00,50974,1869,312.00,8.21,https://cdn.myanimelist.net/images/anime/9/250...,https://myanimelist.net/anime/6864/xxxHOLiC_Rou,...,0,0,0,0,0,0,0,0,0,0
xxxHOLiC Shunmuki,xxxHOLiC Shunmuki,"For the appropriate price, your dearest wish c...","Feb 17, 2009 to Jun 23, 2009",2.00,53279,1813,420.00,8.13,https://cdn.myanimelist.net/images/anime/12/25...,https://myanimelist.net/anime/4918/xxxHOLiC_Sh...,...,0,0,0,0,0,0,0,0,0,0
Üks Uks,Üks Uks,Opening one's inner doors of conflict is the k...,2003,1.00,160,14107,"11,599.00",5.71,https://cdn.myanimelist.net/images/anime/11/71...,https://myanimelist.net/anime/29708/%C3%9Cks_Uks,...,0,0,0,0,0,0,0,0,0,0
ēlDLIVE,ēlDLIVE,Chuuta Kokonose is an orphan who lives with hi...,"Jan 8, 2017 to Mar 26, 2017",12.00,38062,2260,"7,377.00",6.23,https://cdn.myanimelist.net/images/anime/8/823...,https://myanimelist.net/anime/32878/%C4%93lDLIVE,...,0,0,0,0,0,0,0,0,0,0


In [31]:
genres_features = [col for col in one_hot_animes.columns if "genre" in col]
genres_features

['genres',
 "genre_['Action', 'Adventure', 'Cars', 'Comedy', 'Kids', 'Police']",
 "genre_['Action', 'Adventure', 'Cars', 'Comedy', 'Sci-Fi', 'Shounen']",
 "genre_['Action', 'Adventure', 'Cars', 'Sci-Fi']",
 "genre_['Action', 'Adventure', 'Cars']",
 "genre_['Action', 'Adventure', 'Comedy', 'Demons', 'Drama', 'Ecchi', 'Horror', 'Mystery', 'Romance', 'Sci-Fi']",
 "genre_['Action', 'Adventure', 'Comedy', 'Demons', 'Fantasy', 'Magic']",
 "genre_['Action', 'Adventure', 'Comedy', 'Demons', 'Fantasy', 'Martial Arts', 'Shounen', 'Super Power']",
 "genre_['Action', 'Adventure', 'Comedy', 'Demons', 'Fantasy']",
 "genre_['Action', 'Adventure', 'Comedy', 'Demons', 'Shounen', 'Supernatural']",
 "genre_['Action', 'Adventure', 'Comedy', 'Demons', 'Supernatural', 'Martial Arts', 'Shounen']",
 "genre_['Action', 'Adventure', 'Comedy', 'Demons', 'Supernatural', 'Shounen']",
 "genre_['Action', 'Adventure', 'Comedy', 'Demons', 'Supernatural', 'Vampire']",
 "genre_['Action', 'Adventure', 'Comedy', 'Drama', '

In [32]:
anime = 'InuYasha Movie 2: Kagami no Naka no Mugenjo'
one_hot_animes.loc[anime]

title                                            InuYasha Movie 2: Kagami no Naka no Mugenjo
synopsis                                   Inuyasha and company have finally destroyed Na...
aired                                                                           Dec 21, 2002
episodes                                                                                1.00
members                                                                                71989
                                                                 ...                        
genre_['Thriller', 'Mystery', 'Sci-Fi']                                                    0
genre_['Thriller', 'Sci-Fi']                                                               0
genre_['Thriller']                                                                         0
genre_['Vampire']                                                                          0
genre_[]                                                              

In [33]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

if 'genre' in animes.columns:
    animes['genre'] = animes['genre'].astype(str)
    
    df_exploded = animes.assign(genres=animes['genre'].str.split(',')).explode('genre')
    
    df_encoded = pd.get_dummies(df_exploded, columns=['genre'])
    
    one_hot_animes = df_encoded.groupby('title').sum().reset_index()
    
    one_hot_animes.index = one_hot_animes["title"].values

    genres_features = [col for col in one_hot_animes.columns if 'genre_' in col]

    subset_features = one_hot_animes[genres_features]

    subset_features = subset_features.fillna(0)

    scaled_features = StandardScaler().fit_transform(subset_features)

    similarities = cosine_similarity(scaled_features)

    similarities = pd.DataFrame(similarities, columns=subset_features.index, index=subset_features.index)

    similarities.iloc[:50, :50].style.background_gradient(cmap='Purples')

    anime = 'InuYasha Movie 2: Kagami no Naka no Mugenjo'
    n = 10

    top_n_similar = similarities.sort_values(by=anime, ascending=False)[anime].index[1:n+1]

    print(top_n_similar)



Index(['InuYasha Movie 3: Tenka Hadou no Ken',
       'InuYasha Movie 2: Kagami no Naka no Mugenjo',
       'InuYasha Movie 1: Toki wo Koeru Omoi', 'Death Note',
       'Yakushiji Ryouko no Kaiki Jikenbo', 'Death Note: Rewrite',
       'Mahou Shoujo Madoka★Magica Movie 3: Hangyaku no Monogatari',
       'Omoide no Marnie',
       'Danganronpa: Kibou no Gakuen to Zetsubou no Koukousei The Animation',
       'Mouryou no Hako'],
      dtype='object')


In [53]:
print(similarities)

[[1.         0.99982309 0.99744904 ... 0.96682434 0.95839157 0.94763307]
 [0.99982309 1.         0.99857282 ... 0.96847585 0.96167061 0.95122665]
 [0.99744904 0.99857282 1.         ... 0.96970232 0.96855611 0.9573649 ]
 ...
 [0.96682434 0.96847585 0.96970232 ... 1.         0.99359258 0.99431951]
 [0.95839157 0.96167061 0.96855611 ... 0.99359258 1.         0.9956479 ]
 [0.94763307 0.95122665 0.9573649  ... 0.99431951 0.9956479  1.        ]]
